# GKG A/B Benchmark — Sepia Feature Implementation

**Arm A** — Raw LLM: feature spec + raw file dump → code directly  
**Arm B** — GKG pipeline: `map_project → design_feature → implement_feature`

Metrics recorded per run:
- AI turns, prompt tokens, completion tokens, total tokens, elapsed
- Symbol presence, forbidden pattern check, structural checks, diff lines

In [ ]:
import re, time, json, statistics, subprocess, shutil, os, stat
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable
from IPython.display import display, HTML, Markdown

from ollama_client import OllamaClient
from ast_mapper import map_project
from designer import design_feature
from implementer import implement_feature

MODEL     = 'qwen2.5-coder:1.5b'
REPO_URL  = 'https://github.com/MoonFlowww/Sepia.git'
WORK_DIR  = Path('_demo_repos')
TRIALS    = 2   # runs per arm per task

client = OllamaClient(MODEL)
try:
    pong = client.complete('reply: pong', max_tokens=10)
    print(f'ollama ok: {pong.strip()!r}')
except Exception as e:
    print(f'ollama not reachable: {e}')

## 1 · Clone Sepia (fresh)

In [ ]:
def _force_remove(func, path, exc_info):
    os.chmod(path, stat.S_IWRITE)
    func(path)

def clone_fresh(url, work_dir, fresh=True):
    name = url.rstrip('/').split('/')[-1].removesuffix('.git')
    dest = work_dir / name
    if fresh and dest.exists():
        shutil.rmtree(dest, onerror=_force_remove)
    if not dest.exists():
        work_dir.mkdir(parents=True, exist_ok=True)
        r = subprocess.run(['git','clone','--depth=1', url, str(dest)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(r.stderr)
        print(f'cloned → {dest}')
    else:
        print(f'existing: {dest}')
    return dest

REPO = clone_fresh(REPO_URL, WORK_DIR)

def reset_repo():
    subprocess.run(['git','-C',str(REPO),'checkout','--','.'], check=True)
    subprocess.run(['git','-C',str(REPO),'clean','-fd'], check=True)

## 2 · Map GKG once (shared across all tasks)

In [ ]:
client.reset_stats()
t0 = time.time()
GKG = map_project(str(REPO), client)
MAP_STATS = client.stats_summary()
MAP_STATS['elapsed_s'] = time.time() - t0
print(f'GKG: {len(GKG.nodes)} nodes  {len(GKG.edges)} edges  phase={GKG.phase.value}')
print(f'map tokens: prompt={MAP_STATS["prompt_tokens"]}  completion={MAP_STATS["completion_tokens"]}  turns={MAP_STATS["turns"]}')

## 3 · Task definitions

In [ ]:
@dataclass
class Check:
    name: str
    fn: Callable[[str], bool]
    weight: float = 1.0

@dataclass
class Task:
    id: str
    title: str
    feature: str                      # GKG pipeline prompt
    raw_context_files: list[str]      # files to dump in Arm A (relative to REPO)
    expected_symbols: list[str]       # must appear in output
    forbidden: list[str]
    checks: list[Check] = field(default_factory=list)


TASKS = [
    Task(
        id='t1', title='[1] Scientific notation ticks',
        feature=(
            'Add scientific notation support for axis tick labels. '
            'When axis range spans more than 4 orders of magnitude, auto-switch tick labels '
            'to scientific notation (e.g. 1.2e+03). '
            'Add AxisFormat enum (Auto, Fixed, Scientific) to params.hpp. '
            'Add tick_format field to AxisStyle. '
            'Update TickEngine to format labels accordingly.'
        ),
        raw_context_files=[
            'include/sepia/params/params.hpp',
            'include/sepia/rendering/tick_engine.hpp',
            'include/sepia/core/types.hpp',
        ],
        expected_symbols=['AxisFormat', 'Scientific', 'tick_format', 'Auto', 'Fixed'],
        forbidden=['#include <boost>', 'printf('],
        checks=[
            Check('has_enum',         lambda c: 'enum' in c and 'Scientific' in c, 2),
            Check('has_tick_format',  lambda c: 'tick_format' in c),
            Check('has_snprintf_or_to_string', lambda c: 'snprintf' in c or 'to_string' in c or 'ostringstream' in c or 'sprintf' in c),
            Check('has_Auto',         lambda c: 'Auto' in c),
        ],
    ),
    Task(
        id='t2', title='[2] Scatter plot',
        feature=(
            'Add scatter plot support as a new chart type. '
            'Add figure.scatter(x, y, n) method that plots individual markers with no connecting line. '
            'Add ScatterStyle struct with color, marker, marker_size, alpha fields. '
            'Render each point as a marker using existing MarkerStyle primitives. '
            'Add scatter() and scatter_ref() methods to Figure.'
        ),
        raw_context_files=[
            'include/sepia/plot2d/figure.hpp',
            'include/sepia/plot2d/plot_entry.hpp',
            'include/sepia/params/params.hpp',
            'include/sepia/rendering/canvas.hpp',
        ],
        expected_symbols=['scatter', 'ScatterStyle', 'scatter_ref'],
        forbidden=['#include <boost>'],
        checks=[
            Check('has_scatter_method',  lambda c: 'scatter' in c, 2),
            Check('has_ScatterStyle',    lambda c: 'ScatterStyle' in c, 2),
            Check('has_marker_field',    lambda c: 'marker' in c),
            Check('has_scatter_ref',     lambda c: 'scatter_ref' in c),
            Check('no_line_connect',     lambda c: 'scatter' in c and 'draw_line' not in c.split('scatter')[1][:300]),
        ],
    ),
    Task(
        id='t3', title='[3] Pareto frontier + regression on scatter',
        feature=(
            'Add Pareto frontier and regression overlays to scatter plots. '
            'Add compute_pareto(x, y, n) function that returns indices of non-dominated points '
            '(minimizing both x and y). '
            'Add linear_regression(x, y, n) that returns slope and intercept. '
            'Add PlotCommand methods .pareto() and .regression(RegressionType::Linear) '
            'that draw the frontier curve and regression line as overlays. '
            'Add RegressionType enum (Linear, None).'
        ),
        raw_context_files=[
            'include/sepia/plot2d/figure.hpp',
            'include/sepia/plot2d/plot_command.hpp',
            'include/sepia/plot2d/plot_entry.hpp',
            'include/sepia/data/series.hpp',
        ],
        expected_symbols=['compute_pareto', 'linear_regression', 'RegressionType', 'pareto', 'regression'],
        forbidden=['#include <boost>', '#include <eigen>'],
        checks=[
            Check('has_pareto_fn',       lambda c: 'compute_pareto' in c or 'pareto' in c.lower(), 2),
            Check('has_regression_fn',   lambda c: 'linear_regression' in c or 'regression' in c, 2),
            Check('has_RegressionType',  lambda c: 'RegressionType' in c),
            Check('has_slope',           lambda c: 'slope' in c or 'intercept' in c),
            Check('no_external_deps',    lambda c: 'eigen' not in c.lower() and 'armadillo' not in c.lower()),
        ],
    ),
    Task(
        id='t4', title='[4] SIMD-accelerated Wu renderer',
        feature=(
            'Accelerate the Xiaolin Wu anti-aliased line renderer using AVX2 SIMD intrinsics. '
            'Add #ifdef __AVX2__ dispatch: scalar fallback when AVX2 unavailable. '
            'Vectorize pixel blending using _mm256 intrinsics for 4 pixels at a time. '
            'Also vectorize the LTTB triangle-area computation inner loop with __m256d. '
            'Ensure AlignedBuffer uses 32-byte alignment (up from 64 is fine) for AVX2 loads.'
        ),
        raw_context_files=[
            'include/sepia/rendering/canvas.hpp',
            'include/sepia/data/series.hpp',
            'include/sepia/core/memory.hpp',
        ],
        expected_symbols=['__AVX2__', '_mm256', 'AVX2', '__m256d'],
        forbidden=['#include <boost>'],
        checks=[
            Check('has_avx2_guard',    lambda c: '__AVX2__' in c, 2),
            Check('has_mm256',         lambda c: '_mm256' in c, 2),
            Check('has_fallback',      lambda c: '#else' in c or 'else' in c),
            Check('has_immintrin',     lambda c: 'immintrin.h' in c),
            Check('has_m256d',         lambda c: '__m256d' in c),
        ],
    ),
]

print(f'{len(TASKS)} tasks defined')

## 4 · Scoring

In [ ]:
@dataclass
class CodeScore:
    has_symbols: int = 0
    symbols_total: int = 0
    forbidden_clean: bool = True
    check_scores: dict = field(default_factory=dict)
    total: float = 0.0
    max_total: float = 0.0
    diff_added: int = 0
    diff_removed: int = 0

@dataclass
class TokenStats:
    turns: int = 0
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0
    elapsed_s: float = 0.0

@dataclass
class RunResult:
    arm: str
    task_id: str
    code: str           # extracted/combined generated code
    diffs: dict         # {file: diff_str}
    score: CodeScore
    tokens: TokenStats


def score_cpp(code: str, task: Task) -> CodeScore:
    sc = CodeScore(symbols_total=len(task.expected_symbols))
    sc.has_symbols = sum(1 for s in task.expected_symbols if s in code)
    sc.forbidden_clean = not any(f in code for f in task.forbidden)
    for ch in task.checks:
        sc.check_scores[ch.name] = ch.weight if ch.fn(code) else 0.0
    sym_pts    = 2.0 * sc.has_symbols / max(sc.symbols_total, 1)
    forb_pts   = 1.0 if sc.forbidden_clean else 0.0
    check_pts  = sum(sc.check_scores.values())
    sc.total     = sym_pts + forb_pts + check_pts
    sc.max_total = 2.0 + 1.0 + sum(ch.weight for ch in task.checks)
    return sc


def extract_code_blocks(text: str) -> str:
    blocks = re.findall(r'```(?:cpp|c\+\+|c|hpp|h)?\s*\n(.*?)```', text, re.S)
    return '\n\n'.join(b.strip() for b in blocks) if blocks else text.strip()


def diff_stats(diffs: dict) -> tuple[int, int]:
    added = removed = 0
    for d in diffs.values():
        for line in d.splitlines():
            if line.startswith('+') and not line.startswith('+++'):
                added += 1
            elif line.startswith('-') and not line.startswith('---'):
                removed += 1
    return added, removed


def token_stats_from_client(client: OllamaClient) -> TokenStats:
    s = client.stats_summary()
    return TokenStats(
        turns=s['turns'],
        prompt_tokens=s['prompt_tokens'],
        completion_tokens=s['completion_tokens'],
        total_tokens=s['total_tokens'],
        elapsed_s=s['elapsed_s'],
    )

print('scoring ready')

## 5 · Run arms

In [ ]:
RAW_SYS = (
    'You are a C++ expert. Output only fenced C++ code blocks. '
    'No explanation. No prose. Implement the requested feature fully.'
)

def load_files(files: list[str]) -> str:
    parts = []
    for f in files:
        p = REPO / f
        if p.exists():
            parts.append(f'// ===== {f} =====\n{p.read_text(encoding="utf-8", errors="replace")}')
    return '\n\n'.join(parts)


def run_arm_a(task: Task) -> RunResult:
    reset_repo()
    client.reset_stats()
    ctx = load_files(task.raw_context_files)
    prompt = f'EXISTING CODE:\n{ctx}\n\nFEATURE TO IMPLEMENT:\n{task.feature}'
    code_raw = client.complete(prompt, system=RAW_SYS, max_tokens=6000,
                               temperature=0.1, label='raw')
    code = extract_code_blocks(code_raw)
    # build synthetic diff from generated code
    diffs = {'generated.hpp': code} if code else {}
    sc = score_cpp(code, task)
    sc.diff_added, sc.diff_removed = len(code.splitlines()), 0
    tok = token_stats_from_client(client)
    return RunResult('A_raw', task.id, code, diffs, sc, tok)


def run_arm_b(task: Task) -> RunResult:
    reset_repo()
    client.reset_stats()
    bp = design_feature(GKG, task.feature, client)
    diffs, new_contents = implement_feature(GKG, bp, str(REPO), client)
    # combine all generated content for scoring
    code = '\n\n'.join(new_contents.values())
    sc = score_cpp(code, task)
    sc.diff_added, sc.diff_removed = diff_stats(diffs)
    tok = token_stats_from_client(client)
    return RunResult('B_gkg', task.id, code, diffs, sc, tok)


all_results: list[RunResult] = []

for task in TASKS:
    print(f'\n━━ {task.title} ━━')
    for trial in range(TRIALS):
        print(f'  trial {trial+1}/{TRIALS}: ', end='', flush=True)
        try:
            ra = run_arm_a(task)
            all_results.append(ra)
            print(f'A={ra.score.total:.1f}/{ra.score.max_total:.0f}(tok={ra.tokens.total_tokens}) ', end='')
        except Exception as e:
            print(f'A=ERR({e}) ', end='')
            all_results.append(RunResult('A_raw', task.id, '', {}, CodeScore(), TokenStats()))
        try:
            rb = run_arm_b(task)
            all_results.append(rb)
            print(f'B={rb.score.total:.1f}/{rb.score.max_total:.0f}(tok={rb.tokens.total_tokens} turns={rb.tokens.turns})')
        except Exception as e:
            print(f'B=ERR({e})')
            all_results.append(RunResult('B_gkg', task.id, '', {}, CodeScore(), TokenStats()))

print('\n✓ done')

## 6 · Code quality table

In [ ]:
rows = ['| Task | Arm | Score | Symbols | Forbidden✓ | +lines | Time(s) |']
rows.append('|------|-----|-------|---------|------------|--------|---------|')
for task in TASKS:
    for arm, label in [('A_raw','🟢 Raw'), ('B_gkg','🔵 GKG')]:
        runs = [r for r in all_results if r.task_id == task.id and r.arm == arm]
        if not runs: continue
        scores = [r.score.total for r in runs]
        mx = runs[0].score.max_total
        sym = f"{statistics.mean([r.score.has_symbols for r in runs]):.1f}/{runs[0].score.symbols_total}"
        forb = '✓' if all(r.score.forbidden_clean for r in runs) else '✗'
        added = statistics.mean([r.score.diff_added for r in runs])
        elapsed = statistics.mean([r.tokens.elapsed_s for r in runs])
        rows.append(f'| {task.title} | {label} | {statistics.mean(scores):.1f}/{mx:.0f} | {sym} | {forb} | {added:.0f} | {elapsed:.1f} |')
display(Markdown('\n'.join(rows)))

## 7 · Token & turn efficiency table

In [ ]:
rows = ['| Task | Arm | Turns | Prompt tok | Completion tok | Total tok | tok/score |']
rows.append('|------|-----|-------|-----------|----------------|-----------|-----------|')
for task in TASKS:
    for arm, label in [('A_raw','🟢 Raw'), ('B_gkg','🔵 GKG')]:
        runs = [r for r in all_results if r.task_id == task.id and r.arm == arm]
        if not runs: continue
        turns  = statistics.mean([r.tokens.turns for r in runs])
        ptok   = statistics.mean([r.tokens.prompt_tokens for r in runs])
        ctok   = statistics.mean([r.tokens.completion_tokens for r in runs])
        ttok   = statistics.mean([r.tokens.total_tokens for r in runs])
        scores = [r.score.total for r in runs]
        mx     = runs[0].score.max_total
        efficiency = ttok / max(statistics.mean(scores), 0.01)
        rows.append(
            f'| {task.title} | {label} | {turns:.1f} | {ptok:.0f} | {ctok:.0f} | {ttok:.0f} | {efficiency:.0f} |'
        )
# append map cost row
rows.append(f'| *(map, shared)* | 🔵 GKG | {MAP_STATS["turns"]} | {MAP_STATS["prompt_tokens"]} | {MAP_STATS["completion_tokens"]} | {MAP_STATS["prompt_tokens"]+MAP_STATS["completion_tokens"]} | — |')
display(Markdown('\n'.join(rows)))

## 8 · Per-check breakdown

In [ ]:
for task in TASKS:
    print(f'\n{task.title}')
    print('  Check                      A_raw   B_gkg')
    print('  ' + '-'*44)
    check_names = [ch.name for ch in task.checks]
    for cn in check_names:
        a_runs = [r for r in all_results if r.task_id == task.id and r.arm == 'A_raw']
        b_runs = [r for r in all_results if r.task_id == task.id and r.arm == 'B_gkg']
        a_val = statistics.mean([r.score.check_scores.get(cn, 0) for r in a_runs]) if a_runs else 0
        b_val = statistics.mean([r.score.check_scores.get(cn, 0) for r in b_runs]) if b_runs else 0
        print(f'  {cn:<28} {a_val:.2f}    {b_val:.2f}')

## 9 · Aggregate verdict

In [ ]:
print('AGGREGATE QUALITY')
print('─' * 56)
for arm, label in [('A_raw','🟢 Raw LLM'), ('B_gkg','🔵 GKG pipeline')]:
    runs = [r for r in all_results if r.arm == arm]
    if not runs: continue
    norms  = [r.score.total / max(r.score.max_total, 1) for r in runs]
    elapsed = [r.tokens.elapsed_s for r in runs]
    print(f'{label}:  score={statistics.mean(norms):.1%} ±{statistics.pstdev(norms):.1%}  '
          f'time={statistics.mean(elapsed):.1f}s')

print()
print('TOKEN EFFICIENCY')
print('─' * 56)
for arm, label in [('A_raw','🟢 Raw'), ('B_gkg','🔵 GKG')]:
    runs = [r for r in all_results if r.arm == arm]
    if not runs: continue
    ttok   = statistics.mean([r.tokens.total_tokens for r in runs])
    turns  = statistics.mean([r.tokens.turns for r in runs])
    scores = statistics.mean([r.score.total / max(r.score.max_total,1) for r in runs])
    print(f'{label}:  avg_turns={turns:.1f}  avg_total_tok={ttok:.0f}  tok/score={ttok/max(scores,0.01):.0f}')

a_norms = [r.score.total/max(r.score.max_total,1) for r in all_results if r.arm=='A_raw']
b_norms = [r.score.total/max(r.score.max_total,1) for r in all_results if r.arm=='B_gkg']
if a_norms and b_norms:
    lift = statistics.mean(b_norms) - statistics.mean(a_norms)
    print(f'\nGKG lift: {lift:+.1%}')
    a_tok = statistics.mean([r.tokens.total_tokens for r in all_results if r.arm=='A_raw'])
    b_tok = statistics.mean([r.tokens.total_tokens for r in all_results if r.arm=='B_gkg'])
    print(f'Token cost ratio B/A: {b_tok/max(a_tok,1):.2f}x (map cost amortized over {len(TASKS)} tasks)')

## 10 · Side-by-side diffs

In [ ]:
import html as html_mod

def colorize(text, is_diff=False):
    rows = []
    for line in text.splitlines():
        esc = html_mod.escape(line)
        if is_diff:
            if line.startswith('+++') or line.startswith('---'):
                style = 'color:#888'
            elif line.startswith('+'):
                style = 'background:#1a3a1a;color:#7ec87e'
            elif line.startswith('-'):
                style = 'background:#3a1a1a;color:#c87e7e'
            elif line.startswith('@@'):
                style = 'color:#7eafc8'
            else:
                style = 'color:#ccc'
        else:
            style = 'color:#eee'
        rows.append(f'<div style="{style};white-space:pre">{esc}</div>')
    return '\n'.join(rows)

box = 'font-family:monospace;font-size:11px;background:#1e1e1e;padding:10px;border-radius:6px;max-height:350px;overflow-y:auto'

for task in TASKS:
    a_runs = [r for r in all_results if r.task_id == task.id and r.arm == 'A_raw']
    b_runs = [r for r in all_results if r.task_id == task.id and r.arm == 'B_gkg']
    a_code = a_runs[0].code if a_runs else '(no result)'
    b_diff = '\n'.join(b_runs[0].diffs.values()) if b_runs and b_runs[0].diffs else '(no diff)'
    sa = a_runs[0].score if a_runs else CodeScore()
    sb = b_runs[0].score if b_runs else CodeScore()
    ta = a_runs[0].tokens if a_runs else TokenStats()
    tb = b_runs[0].tokens if b_runs else TokenStats()
    display(HTML(f'''
    <h3>{task.title}</h3>
    <div style="display:flex;gap:8px">
      <div style="flex:1;border:1px solid #555;padding:8px;border-radius:6px">
        <b>🟢 Raw</b> score={sa.total:.1f}/{sa.max_total:.0f} tok={ta.total_tokens} turns={ta.turns}<br>
        <div style="{box}">{colorize(a_code)}</div>
      </div>
      <div style="flex:1;border:1px solid #555;padding:8px;border-radius:6px">
        <b>🔵 GKG</b> score={sb.total:.1f}/{sb.max_total:.0f} tok={tb.total_tokens} turns={tb.turns}<br>
        <div style="{box}">{colorize(b_diff, is_diff=True)}</div>
      </div>
    </div>'''))